# Stage 4: plots

Draws the final figures for each condition (Nyquist, Bode, stacked DRT, Arrhenius)
and, across all conditions at once, the Brouwer $p_{\text{O}_2}$ diagram. Stage 3
produced numbers per spectrum; this stage is where they become something you can
read a trend off.

**Reads:** `{sample_id}/Results/{condition}/stage3_drt.xlsx` · `stage3_fit.xlsx` · `ISM validation/{condition}/*.ism`
**Writes:** `{sample_id}/Results/{condition}/{DRT,Nyquist-Bode,Arrhenius}/` · `Results/pO2/`


## Quick links

- [Configuration](#configuration): `SAMPLE_ID`, `CONDITION_FILTER`, `L_m`, `D_m`, `PLOT_WINDOWS`, `DRT_TAU_MAX`
- [Step 1: Figures per condition](#step-1-figures-per-condition): stacked DRT, Nyquist, Bode, Arrhenius
- [Step 1b: DRT window](#step-1b-drt-window): move the $\tau$ axis of the stacked DRT figure
- [Step 2: Brouwer p(O2)](#step-2-brouwer-po2-all-conditions): $\sigma$ against $p_{\text{O}_2}$, every condition on one plot
- [Step 2b: Brouwer peak selector](#step-2b-brouwer-select-peak-and-temperatures): interactive peak and temperature filter
- [Step 3: transference numbers](#step-3-ionicelectronic-decomposition-transference-numbers): the $t_{\text{ion}}$ and $t_{\text{elect}}$ split from the Brouwer fit
- [Output summary](#output-summary): exported figure paths

**Prerequisite:** run `stage3_drt.ipynb` on this sample first. It stores the pellet
geometry (`L_m`, `D_m`) in `session.json` and writes the `stage3_fit.xlsx` and
`stage3_drt.xlsx` files this stage reads.

**Workflow:** edit the config cell, run the plot cell, then use the Step 2b peak
selector without scrolling back. Per-condition axis crops go in `PLOT_WINDOWS`
(`session.json`, `stage4_params`); the keys are condition-level (`z_max`, `freq_min`
and so on), and per-temperature sub-keys survive a round-trip but are not applied.


## Configuration

**Mode switch** `PARAM_MODE` (top of the cell below), hand-edited and the cell re-run to switch:

- `"lock"`: read-only reproduction of the saved calibration, nothing writes back.
- `"continue"`: config cell is the base, starting values load from `session.json` when present, widget/Apply edits merge-save.
- `"reset"`: deliberately ignore `session.json`, start from the literals below, next save overwrites the saved history.

Full semantics: `docs/STAGES.md`, "Configuration modes".


In [ ]:
import sys
from pathlib import Path
from pipeline.interactive import select_sample, param_source_banner, dialed
from pipeline.session import load_sample, update_sample_guarded

NOTEBOOK_DIR = Path.cwd()

sample_id = select_sample(NOTEBOOK_DIR, show_list=True)

_cfg = load_sample(sample_id)

# Recorded by stage 1: False means the lambda probe was off, so the stored
# pO2_mean is an idle reading. Absent means the probe was on.
PO2_PROBE = bool(_cfg.get("pO2_probe", True))
if not PO2_PROBE:
    print("p(O2) probe: OFF - pressures are hidden and the p(O2) analyses are skipped")

# Optional: process only specific conditions (leave empty [] to process ALL)
CONDITION_FILTER = []
condition_filter = CONDITION_FILTER

# Sample geometry - loaded from session.json (written by Stage 3)
L_m = _cfg.get("L_m")
D_m = _cfg.get("D_m")

if L_m is None or D_m is None:
    raise ValueError(
        f"Sample geometry (L_m, D_m) not found in session.json for '{sample_id}'.\n"
        "Run stage3_drt.ipynb first: its configuration cell asks for pellet "
        "thickness L and diameter D and saves them for Stage 4."
    )

# lock: read-only, no write.
# continue: read session.json if present, edits merge-save.
# reset: ignores session.json, starts from literals below, next save overwrites.
# Full semantics: docs/STAGES.md, "Configuration modes".
PARAM_MODE = "continue"


def _update_session(**fields) -> bool:
    """Merge-save to session.json. No-op (returns False) in lock mode."""
    return update_sample_guarded(sample_id, PARAM_MODE, **fields)

# Starting values load from session.json unless reset: re-run must never
# silently wipe saved tuning.
_p4 = {} if PARAM_MODE == "reset" else _cfg.get("stage4_params", {})
param_source_banner(PARAM_MODE, "Stage 4")

DRT_TAU_MAX      = _p4.get("DRT_TAU_MAX",      0.1)
# None keeps the left edge this plot has always drawn (4e-08). Set it from the
# Step 1b panel when a sample measured to lower frequencies needs a wider frame.
DRT_TAU_MIN      = _p4.get("DRT_TAU_MIN",      None)
BROUWER_PEAK_ID  = _p4.get("BROUWER_PEAK_ID",  1)
BROUWER_TEMPS    = _p4.get("BROUWER_TEMPS",     None)
# Reference slope guides drawn on the Brouwer diagrams; any subset of
# "-1/4", "-1/6", "0", "+1/6", "+1/4"
BROUWER_SLOPES   = _p4.get("BROUWER_SLOPES",   ["-1/4", "-1/6", "0", "+1/6", "+1/4"])
# Exclude T below this value [°C] from all Arrhenius fits (None = use all):
# set it when peak identity is not resolved at low T (e.g. N_peaks drops to 3)
ARRHENIUS_T_MIN  = _p4.get("ARRHENIUS_T_MIN",   None)
# Peaks whose series sum forms the HF block in the single-panel sigma
# Arrhenius (e.g. [1, 2]); the sum is drawn over the full T range while
# the separated branches respect ARRHENIUS_T_MIN. None disables the figure.
ARRHENIUS_SUM_PEAKS = _p4.get("ARRHENIUS_SUM_PEAKS", None)
# Brouwer exponent x for the ionic/electronic decomposition (0.25 in the
# dilute defect regime; 1/6 in other regimes)
TRANSF_EXPONENT  = _p4.get("TRANSF_EXPONENT",   1/4)
# Peaks shown in the transference figures (None = all). The table always
# covers every peak; restrict the FIGURES to transport processes (e.g. [1, 2])
# because t_ion is physically meaningful only for bulk/GB, not electrodes.
TRANSF_PEAK_IDS  = _p4.get("TRANSF_PEAK_IDS", None)
# Physics behind the three settings above: docs/STAGES.md, HF-block sum / Ionic/electronic decomposition (per isotherm).
# session.json stringifies dict keys: restore per-T int keys, keep
# condition-level string keys (z_max, freq_min, ...) as-is.
PLOT_WINDOWS = {
    cond: {(int(k) if isinstance(k, str) and k.isdigit() else k): v
           for k, v in win.items()}
    for cond, win in _p4.get("PLOT_WINDOWS", {}).items()
}

# Manual (condition, T) validity chosen in stage3 ("Validity selection" cell).
# True = figures skip deselected points; the xlsx files on disk stay complete.
USE_STAGE3_SELECTION = _p4.get("USE_STAGE3_SELECTION", True)
STAGE3_VALID = _cfg.get("stage3_valid", {})

def _apply_stage3_valid(df, condition):
    """Drop rows whose T_nominal was deselected in stage3 for this condition."""
    ts = STAGE3_VALID.get(condition) if USE_STAGE3_SELECTION else None
    if ts is None or "T_nominal" not in df.columns:
        return df
    ts = {int(t) for t in ts}
    return df[df["T_nominal"].astype(int).isin(ts)].reset_index(drop=True)

def _stage4_params() -> dict:
    return {
        "DRT_TAU_MAX":      DRT_TAU_MAX,
        "DRT_TAU_MIN":      DRT_TAU_MIN,
        "BROUWER_PEAK_ID":  BROUWER_PEAK_ID,
        "BROUWER_TEMPS":    BROUWER_TEMPS,
        "BROUWER_SLOPES":   BROUWER_SLOPES,
        "ARRHENIUS_T_MIN":  ARRHENIUS_T_MIN,
        "ARRHENIUS_SUM_PEAKS": ARRHENIUS_SUM_PEAKS,
        "TRANSF_EXPONENT":  TRANSF_EXPONENT,
        "TRANSF_PEAK_IDS":  TRANSF_PEAK_IDS,
        "PLOT_WINDOWS":     PLOT_WINDOWS,
        "USE_STAGE3_SELECTION": USE_STAGE3_SELECTION,
    }

_update_session(stage4_params=_stage4_params())


def _nyquist_xylim(window: dict) -> tuple:
    xlim = ylim = None
    z_min = window.get("z_min", 0)
    if "z_max" in window:
        xlim = (z_min, window["z_max"])
        ylim = (0, window["z_max"])
    return xlim, ylim


def _bode_freqlim(window: dict) -> tuple | None:
    fmin = window.get("freq_min")
    fmax = window.get("freq_max")
    if fmin is None and fmax is None:
        return None
    return (fmin if fmin is not None else 1e-3, fmax if fmax is not None else 1e9)

In [ ]:
# inline backend: more reliable than ipympl with ipywidgets panels.
get_ipython().run_line_magic("matplotlib", "inline")  # type: ignore[name-defined]

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline.ingest import load_ism, load_csv_spectrum
from pipeline.drt import clip_spectrum
from pipeline.quality import strip_inductive
from pipeline.plots import (
    apply_pub_style,
    plot_drt_stacked,
    plot_nyquist_multipanel,
    plot_bode,
    plot_arrhenius_panel,
    plot_arrhenius_sigma,
    plot_brouwer,
    build_arrhenius_results,
)

# Apply publication style once
apply_pub_style()

sample_dir   = NOTEBOOK_DIR / sample_id
RESULTS_BASE = sample_dir / "Results"

# Collect conditions that have completed Stage 3
all_conditions = sorted([
    d.name for d in RESULTS_BASE.iterdir()
    if d.is_dir()
    and (d / "stage3_fit.xlsx").exists()
    and (d / "stage3_drt.xlsx").exists()
])

conditions = (
    [c for c in all_conditions if c in condition_filter]
    if condition_filter else all_conditions
)

print(f"Sample     : {sample_id}")
print(f"Geometry   : L = {L_m*1e3:.3f} mm  D = {D_m*1e3:.3f} mm")
print(f"Conditions with Stage 3 output ({len(conditions)}):")
for c in conditions:
    print(f"  {c}")


def _condition_window(condition: str) -> dict:
    """Condition-level PLOT_WINDOWS entry: only top-level string keys (skip per-T int keys)."""
    cw = PLOT_WINDOWS.get(condition, {})
    return {k: v for k, v in cw.items() if isinstance(k, str)}
try:
    import ipywidgets as W
    from IPython.display import display as _display
    _HAS_WIDGETS = True
except Exception as _exc:
    print(f"[INFO] ipywidgets not installed ({_exc}); control panels disabled.")
    _HAS_WIDGETS = False


## Step 1: figures per condition

For each condition: loads the validated spectra and the Zarc parameters from Stage 3,
then draws the stacked DRT, Nyquist, Bode and Arrhenius figures. Every fitted peak
appears in the Arrhenius panels, and the $R^2(\tau)$ of each fit is reported in the
summary table.

All figures are written to `Results/{condition}/` as PNG and PDF.

*Arrhenius panels: `Ea_cond` from $\ln(\sigma T)$ against $1/T$ (long-range transport),
`Ea_pol` from $\ln(\tau)$ (local relaxation, not expected to be linear for electrode
processes), `Ea_C` from $\ln(C_\text{eff})$, which comes out as the difference
`Ea_pol` minus `Ea_cond`. Derivations: `docs/MATHEMATICS.md` section 4.*


In [ ]:
# Collect all peak data across conditions (used later for the Brouwer diagram)
all_peaks_df_list = []

# Cache of in-memory data per condition; used by the live control panel below
_plot_cache: dict[str, dict] = {}

for condition in conditions:
    print(f"\n{'='*70}")
    print(f"Condition: {condition}")
    print(f"{'='*70}")

    res_dir  = RESULTS_BASE / condition
    val_dir  = sample_dir / "ISM validation" / condition
    csv_dir  = sample_dir / "input_spectra" / condition   # CSV entry mode

    # Load Stage 3 outputs
    _fit_sheets    = pd.read_excel(res_dir / "stage3_fit.xlsx",  sheet_name=["Peaks", "Summary"])
    df_fit_peaks, df_fit_summary = _fit_sheets["Peaks"], _fit_sheets["Summary"]
    df_drt_spectra = pd.read_excel(res_dir / "stage3_drt.xlsx",  sheet_name="DRT_Spectra")
    df_kk_sel      = pd.read_excel(res_dir / "stage2_kk.xlsx",   sheet_name="Selected")

    if USE_STAGE3_SELECTION and STAGE3_VALID.get(condition) is not None:
        df_fit_peaks   = _apply_stage3_valid(df_fit_peaks,   condition)
        df_fit_summary = _apply_stage3_valid(df_fit_summary, condition)
        df_drt_spectra = _apply_stage3_valid(df_drt_spectra, condition)
        df_kk_sel      = _apply_stage3_valid(df_kk_sel,      condition)
        print(f"  stage3 selection: keeping T = "
              f"{sorted(int(t) for t in STAGE3_VALID[condition])}")

    # Collect for Brouwer aggregation
    all_peaks_df_list.append(df_fit_peaks)

    temps_available = sorted(df_fit_summary["T_nominal"].unique())
    print(f"  Temperatures: {[int(t) for t in temps_available]}")

    # Load ISM data (validated, same files used in Stage 3)
    records    = {}   # {T_nominal: (freq, Z_re, Z_im)}
    fit_params = {}   # {T_nominal: {R0, R, tau, alpha}}

    for _, row in df_kk_sel.iterrows():
        T_nom  = int(row["T_nominal"])
        fname  = row["file"]
        f_min  = row["f_min_cut"] if pd.notna(row.get("f_min_cut")) else None
        f_max  = row["f_max_cut"] if pd.notna(row.get("f_max_cut")) else None

        ism_path = val_dir / fname
        if not ism_path.exists() and (csv_dir / fname).exists():
            ism_path = csv_dir / fname
        if not ism_path.exists():
            print(f"  [WARN] Not found: {fname}; skipping T={T_nom}")
            continue

        rec  = (load_csv_spectrum(ism_path)
                if ism_path.suffix.lower() in (".csv", ".txt")
                else load_ism(ism_path))
        # Figures show only physically valid points: the same Z' >= 0 / Z'' >= 0
        # criterion applied before Lin-KK (no passive circuit can produce them;
        # verified not to affect the fitted parameters)
        _fs, _zrs, _zis, _n_str = strip_inductive(rec.freq, rec.Z_re, rec.Z_im)
        freq, Z_re, Z_im = clip_spectrum(_fs, _zrs, _zis, f_min, f_max)
        records[T_nom] = (freq, Z_re, Z_im)

        sub = df_fit_peaks[df_fit_peaks["T_nominal"] == T_nom]
        if not sub.empty:
            sum_row = df_fit_summary[df_fit_summary["T_nominal"] == T_nom]
            R0_val  = float(sum_row["R0"].iloc[0]) if not sum_row.empty else None
            fit_params[T_nom] = {
                "R0":    R0_val,
                "R":     sub["R_i"].values.tolist(),
                "tau":   sub["tau_i"].values.tolist(),
                "alpha": sub["alpha_i"].values.tolist(),
            }

    _plot_cache[condition] = {
        "records":        records,
        "fit_params":     fit_params,
        "df_drt_spectra": df_drt_spectra,
        "df_fit_peaks":   df_fit_peaks,
        "df_fit_summary": df_fit_summary,
        "res_dir":        res_dir,
    }

    # Output sub-directories
    drt_dir  = res_dir / "DRT"
    nq_dir   = res_dir / "Nyquist-Bode"
    arr_dir  = res_dir / "Arrhenius"

    # Resolve condition-level plot window (None = full range)
    cw = _condition_window(condition)
    nyq_xlim, nyq_ylim = _nyquist_xylim(cw)
    bode_freq          = _bode_freqlim(cw)

    # 1. DRT stacked plot
    print("  -> DRT stacked plot")
    if not df_drt_spectra.empty:
        fig_drt = plot_drt_stacked(
            df_spectra   = df_drt_spectra,
            condition    = condition,
            save_dir     = drt_dir,
            tau_max      = DRT_TAU_MAX,
            tau_min      = DRT_TAU_MIN,
            df_peaks     = df_fit_peaks,
            show_pO2     = PO2_PROBE,
        )
        plt.show()
        plt.close(fig_drt)
    else:
        print("    [SKIP] No DRT spectra data.")

    # 2. Nyquist overlay (with optional condition-level crop)
    print("  -> Nyquist overlay" + (f"  [window z_max={cw.get('z_max')} kOhm]" if "z_max" in cw else ""))
    if records:
        fig_nq = plot_nyquist_multipanel(
            records    = records,
            fit_params = fit_params,
            condition  = condition,
            save_dir   = nq_dir,
            xlim       = nyq_xlim,
            ylim       = nyq_ylim,
            df_peaks   = df_fit_peaks,
            show_pO2   = PO2_PROBE,
        )
        plt.show()
        plt.close(fig_nq)

    # 3. Bode plot (with optional condition-level freq window)
    print("  -> Bode plot" + (f"  [freq window {bode_freq}]" if bode_freq else ""))
    if records:
        fig_bode = plot_bode(
            records    = records,
            fit_params = fit_params,
            condition  = condition,
            save_dir   = nq_dir,
            freq_lim   = bode_freq,
            df_peaks   = df_fit_peaks,
            show_pO2   = PO2_PROBE,
        )
        plt.show()
        plt.close(fig_bode)

    # 4. Arrhenius 2x2 panel
    print("  -> Arrhenius panel")
    if not df_fit_peaks.empty:
        fig_arr, results_all = plot_arrhenius_panel(
            df_peaks     = df_fit_peaks,
            L_m          = L_m,
            D_m          = D_m,
            condition    = condition,
            save_dir     = arr_dir,
            t_min        = ARRHENIUS_T_MIN,
            show_pO2     = PO2_PROBE,
        )
        plt.show()
        plt.close(fig_arr)

        # Print activation energy summary table
        print(f"\n  Activation energies for {condition}")
        print(f"  {'Peak':<10} {'Ea_cond (eV)':<18} {'Ea_pol (eV)':<18} "
              f"{'Ea_C (eV)':<18} {'R2_cond':<10} {'R2_pol':<10} {'R2_C':<10}")
        print(f"  {'-'*94}")
        for r in results_all:
            def _fmt(v, e):
                return f"{v:.3f}+/-{e:.3f}" if not (np.isnan(v) or np.isnan(e)) else "N/A"
            def _r2(v):
                return f"{v:.4f}" if not np.isnan(v) else "N/A"
            print(f"  {r['Peak']:<10} {_fmt(r['Ea_cond'], r['Ea_cond_err']):<18} "
                  f"{_fmt(r['Ea_pol'], r['Ea_pol_err']):<18} "
                  f"{_fmt(r['Ea_C'], r['Ea_C_err']):<18} "
                  f"{_r2(r['R2_cond']):<10} {_r2(r['R2_pol']):<10} {_r2(r['R2_C']):<10}")
        print()

    # 5. HF-block sigma Arrhenius: separated branches + series sum
    if ARRHENIUS_SUM_PEAKS and not df_fit_peaks.empty:
        print("  -> HF-block sigma Arrhenius")
        fig_sig = plot_arrhenius_sigma(
            df_peaks     = df_fit_peaks,
            L_m          = L_m,
            D_m          = D_m,
            condition    = condition,
            save_dir     = arr_dir,
            t_min        = ARRHENIUS_T_MIN,
            sum_peak_ids = ARRHENIUS_SUM_PEAKS,
            show_pO2     = PO2_PROBE,
        )
        if fig_sig is not None:
            plt.show()
            plt.close(fig_sig)

    print(f"  Figures saved in {res_dir.relative_to(NOTEBOOK_DIR)}")

print(f"\n{'='*70}")
print(f"Per-condition figures complete; {len(conditions)} condition(s).")

## Step 1b: DRT window

Moves the $\tau$ axis of the stacked DRT figure, for a sample measured down to lower
frequencies whose peaks sit outside the default frame. Framing only: no point is
dropped and no curve is renormalized, so the peaks keep the heights they have in the
saved figures.

**Preview** redraws the selected condition and writes nothing. **Save window** stores
the two values in `session.json`; re-run Step 1 to redraw the saved figures with them.
One window applies to every condition of the sample.

The upper limit keeps a second job it has always had: points above it are dropped
before each curve is normalized, so moving that one can change the curve heights.


In [ ]:
# DRT window: move the tau axis of the stacked DRT figure. The lower limit frames
# only, so no point is dropped and no curve is renormalized.
# Design rule (same as the other panels): no W.Output. The plot is a W.Image and the
# status a W.HTML, both value-replaced, so a redraw updates the SAME image in place.
from io import BytesIO as _BytesIO_DRT

from IPython.display import display as _display_drt
from pipeline.interactive import pre_html as _pre_drt

# 4e-08 is the edge this plot has always drawn (the label position, 5e-08, times
# 0.8), so the box opens on the frame in force rather than on a rounder number.
_TAU_MIN_SHOWN = DRT_TAU_MIN if DRT_TAU_MIN is not None else 4e-8

if _HAS_WIDGETS and _plot_cache:
    _w_dcond = W.Dropdown(options=sorted(_plot_cache), description="Cond:",
                          layout=W.Layout(width="420px"))
    _w_tmin  = W.BoundedFloatText(value=_TAU_MIN_SHOWN, min=1e-12, max=1e3,
                                  description="tau min [s]:",
                                  style={"description_width": "90px"},
                                  layout=W.Layout(width="230px"))
    _w_tmax  = W.BoundedFloatText(value=DRT_TAU_MAX, min=1e-12, max=1e3,
                                  description="tau max [s]:",
                                  style={"description_width": "90px"},
                                  layout=W.Layout(width="230px"))
    _w_dgo   = W.Button(description="Preview", button_style="primary",
                        layout=W.Layout(width="130px"),
                        tooltip="Redraw the selected condition with this window; writes nothing")
    _w_dsave = W.Button(description="\U0001F4BE Save window", button_style="success",
                        layout=W.Layout(width="170px"),
                        tooltip="Store the window in session.json; re-run Step 1 to redraw the saved figures")
    _dimg = W.Image(format="png", layout=W.Layout(width="100%", max_width="520px"))
    _dmsg = W.HTML()

    def _drt_window() -> tuple:
        """The window in the boxes, or (None, None) when it is not an interval."""
        lo, hi = float(_w_tmin.value), float(_w_tmax.value)
        return (lo, hi) if lo < hi else (None, None)

    def _on_drt_preview(_b=None):
        lo, hi = _drt_window()
        if lo is None:
            _dmsg.value = _pre_drt("tau min must be below tau max; nothing drawn.")
            return
        cond  = _w_dcond.value
        cache = _plot_cache[cond]
        fig = plot_drt_stacked(
            df_spectra = cache["df_drt_spectra"],
            condition  = cond,
            save_dir   = cache["res_dir"] / "DRT",
            tau_max    = hi,
            tau_min    = lo,
            df_peaks   = cache["df_fit_peaks"],
            show_pO2   = PO2_PROBE,
            save       = False,
        )
        _buf = _BytesIO_DRT()
        fig.savefig(_buf, format="png", dpi=110)
        plt.close(fig)
        _dimg.value = _buf.getvalue()
        _dmsg.value = _pre_drt(f"preview {cond}: tau from {lo:g} to {hi:g} s (nothing written)")
    _w_dgo.on_click(_on_drt_preview)

    def _on_drt_save(_b):
        global DRT_TAU_MIN, DRT_TAU_MAX
        lo, hi = _drt_window()
        if lo is None:
            _dmsg.value = _pre_drt("tau min must be below tau max; nothing saved.")
            return
        DRT_TAU_MIN, DRT_TAU_MAX = dialed(lo), dialed(hi)
        if _update_session(stage4_params=_stage4_params()):
            _dmsg.value = _pre_drt(
                f"saved tau window {DRT_TAU_MIN:g} to {DRT_TAU_MAX:g} s.\n"
                "Re-run Step 1 to redraw every saved figure with it.")
        else:
            _dmsg.value = _pre_drt("PARAM_MODE = lock: nothing was saved.")
    _w_dsave.on_click(_on_drt_save)

    _display_drt(W.VBox([_w_dcond,
                         W.HBox([_w_tmin, _w_tmax, _w_dgo, _w_dsave]),
                         _dmsg, _dimg]))
elif not _plot_cache:
    print("[INFO] No figures in memory; run Step 1 first.")


## Step 2: Brouwer p(O2); all conditions

Collects peak `BROUWER_PEAK_ID` from **all** conditions and plots
$\log_{10}(\sigma)$ against $\log_{10}(p_{\text{O}_2})$.

Each symbol marks a temperature (400-600 °C). The slope guides at $-1/4$, $-1/6$,
$0$ (plateau), $+1/6$ and $+1/4$ are there to read the defect regime off the plot
(see `BROUWER_SLOPES`).

> **Note**: the diagram means something only if the same `peak_id` is the same physical
> process in every condition. Check that with the $C_\text{eff}$ magnitude and the
> Arrhenius behavior before reading any slope off it.


In [ ]:
if all_peaks_df_list:
    df_all_peaks = pd.concat(all_peaks_df_list, ignore_index=True)

    # skip Brouwer if pO2 data is not available (CSV/TXT entry mode)
    _has_pO2 = (
        PO2_PROBE
        and "pO2_mean" in df_all_peaks.columns
        and df_all_peaks["pO2_mean"].notna().any()
    )
    if not _has_pO2:
        print("[SKIP] Brouwer diagram: no usable pO2 data. It needs furnace-log "
              "data (stage 0 + stage 1) and a lambda probe that was switched on.")
    else:
        peak1_data = df_all_peaks[df_all_peaks["peak_id"] == BROUWER_PEAK_ID]
        n_cond_p1  = peak1_data["condition"].nunique() if not peak1_data.empty else 0
        print(f"Peak {BROUWER_PEAK_ID} data found in {n_cond_p1} condition(s) of {len(conditions)} total.")

        if n_cond_p1 < 2:
            print("  [SKIP] Brouwer diagram requires 2 or more conditions. "
                  "Run Stage 3 on more atmospheric conditions first.")
        else:
            brouwer_dir = RESULTS_BASE / "pO2"
            # Every peak gets its official diagram, all with the same
            # BROUWER_TEMPS filter: stale per-peak figures from older runs
            # would otherwise survive on disk and mix analysis vintages.
            for _pid in sorted(int(p) for p in df_all_peaks["peak_id"].unique()):
                _n_cond = df_all_peaks[df_all_peaks["peak_id"] == _pid]["condition"].nunique()
                if _n_cond < 2:
                    print(f"  [SKIP] Peak {_pid}: present in {_n_cond} condition(s)")
                    continue
                fig_brouwer = plot_brouwer(
                    df_all        = df_all_peaks,
                    save_dir      = brouwer_dir,
                    sample_name   = sample_id,
                    peak_id       = _pid,
                    temps_to_plot = BROUWER_TEMPS,
                    add_slopes    = True,
                    slopes        = tuple(BROUWER_SLOPES),
                )
                if _pid == BROUWER_PEAK_ID:
                    # raw Figure (not pyplot): shown explicitly, nothing to close
                    display(fig_brouwer)
            print(f"Brouwer diagrams saved in {brouwer_dir.relative_to(NOTEBOOK_DIR)}")

            print(f"\nPeak {BROUWER_PEAK_ID} data used for Brouwer diagram:")
            tbl = (
                peak1_data[["condition", "T_nominal", "pO2_mean", "R_i", "sigma_Sm_i"]]
                .sort_values(["T_nominal", "pO2_mean"])
                .reset_index(drop=True)
            )
            tbl["lg_pO2"]   = tbl["pO2_mean"].apply(
                lambda x: f"{np.log10(x):.3f}" if x > 0 else "N/A")
            tbl["lg_sigma"] = tbl["sigma_Sm_i"].apply(
                lambda x: f"{np.log10(x/100):.3f}" if x > 0 else "N/A")
            display(tbl[["condition", "T_nominal", "lg_pO2", "lg_sigma", "R_i"]])
else:
    print("No conditions processed; nothing to aggregate.")

## Step 2b: Brouwer: select peak and temperatures

Pick a **peak** and, if you want, a subset of temperatures, then press
**↻ Replot Brouwer**. The figure `Brouwer_Peak{N}_{sample}.{png,pdf}` is written to
`Results/pO2/`.

Step 2 has to have run first: this panel reads `df_all_peaks` from memory.


In [ ]:
# Brouwer selector: replot the p(O2) diagram for a chosen peak / temperatures /
# conditions from df_all_peaks (Step 2, in memory); never recomputes fits.
# Overwrites the canonical Brouwer_Peak{N}_{sample}.{png,pdf} on disk.
#
# Design rule (same as the stage-3 panels): NO W.Output. The plot is a W.Image and
# the status line a W.HTML, both value-replaced, so each Replot updates the SAME
# image in place instead of appending a new diagram under it (which is what clearing
# an Output inside a button callback does in VSCode).
from io import BytesIO as _BytesIO

_HAS_WIDGETS_BR = _HAS_WIDGETS


from pipeline.interactive import pre_html as _pre


if (_HAS_WIDGETS_BR and PO2_PROBE and ("df_all_peaks" in dir())
        and not df_all_peaks.empty
        and df_all_peaks["condition"].nunique() >= 2):
    _brouwer_dir = RESULTS_BASE / "pO2"
    _peak_ids  = sorted(int(p) for p in df_all_peaks["peak_id"].unique())
    _all_temps = sorted(int(t) for t in df_all_peaks["T_nominal"].unique())
    _all_conds = sorted(df_all_peaks["condition"].unique())

    w_peak = W.Dropdown(options=_peak_ids,
                        value=BROUWER_PEAK_ID if BROUWER_PEAK_ID in _peak_ids else _peak_ids[0],
                        description="Peak:", layout=W.Layout(width="180px"))
    w_temps = W.SelectMultiple(options=_all_temps, value=tuple(_all_temps),
                               description="T [°C]:", rows=min(9, len(_all_temps)),
                               layout=W.Layout(width="180px"))
    # Condition selector; deselect e.g. the Ar condition to drop it from the diagram.
    w_conds = W.SelectMultiple(options=_all_conds, value=tuple(_all_conds),
                               description="Cond:", rows=min(6, len(_all_conds)),
                               layout=W.Layout(width="440px"))
    _slope_opts = ["-1/4", "-1/6", "0", "+1/6", "+1/4"]
    w_slopes = W.SelectMultiple(options=_slope_opts,
                                value=tuple(s for s in BROUWER_SLOPES if s in _slope_opts),
                                description="Slopes:", rows=5,
                                layout=W.Layout(width="180px"))
    w_br_go = W.Button(description="↻ Replot Brouwer", button_style="primary",
                       layout=W.Layout(width="200px"))
    txt_br = W.HTML()   # status line
    img_br = W.Image(format="png", layout=W.Layout(width="100%", max_width="770px"))
    _br_busy = [False]

    def _brouwer_impl():
        peak_id = int(w_peak.value)
        sel_T   = [int(t) for t in w_temps.value]
        temps   = sel_T if (sel_T and len(sel_T) != len(_all_temps)) else None
        sel_C   = list(w_conds.value) or _all_conds
        df_sel  = df_all_peaks[df_all_peaks["condition"].isin(sel_C)]
        sub     = df_sel[df_sel["peak_id"] == peak_id]
        n_cond  = sub["condition"].nunique() if not sub.empty else 0
        if n_cond < 2:
            txt_br.value = _pre(f"[SKIP] Peak {peak_id} present in only {n_cond} selected "
                                "condition(s); Brouwer needs >= 2. Select more conditions.")
            return
        fig = plot_brouwer(
            df_all=df_sel, save_dir=_brouwer_dir,
            sample_name=sample_id, peak_id=peak_id,
            temps_to_plot=temps,
            slopes=tuple(w_slopes.value),
        )
        buf = _BytesIO()
        fig.savefig(buf, format="png", dpi=110)
        img_br.value = buf.getvalue()
        txt_br.value = _pre(
            f"Saved -> {(_brouwer_dir / f'Brouwer_Peak{peak_id}_{sample_id}')}.png/.pdf"
            "  (overwrites the Step-2 figure)\n"
            f"  peak={peak_id}  temps={'all' if temps is None else temps}\n"
            f"  conditions ({n_cond}): {sel_C}")

    def _on_brouwer(_btn):
        # ignore clicks queued while a replot is already running
        if _br_busy[0]:
            return
        _br_busy[0] = True
        w_br_go.disabled = True
        try:
            _brouwer_impl()
        except Exception:
            import traceback
            txt_br.value = _pre("Replot failed:\n" + traceback.format_exc())
        finally:
            _br_busy[0] = False
            w_br_go.disabled = False
    w_br_go.on_click(_on_brouwer)

    _display(W.VBox([W.HBox([w_peak, w_temps, w_conds, w_slopes]), w_br_go, txt_br, img_br]))
elif _HAS_WIDGETS_BR:
    print("[INFO] Brouwer selector unavailable: it needs Step 2 to have run, a "
          "lambda probe that was on, and 2 or more conditions.")


## Step 3: Ionic/electronic decomposition (transference numbers)

**Code:** `pipeline/plots.py::fit_transference` (derivation: `docs/MATHEMATICS.md` section 5).

Per-temperature Patterson fit of the Brouwer data, one isotherm at a time:

$\sigma(p_{\text{O}_2}) = \sigma_{\text{ion}} + \sigma_{p\text{-type}}\, p_{\text{O}_2}^{+x} + \sigma_{n\text{-type}}\, p_{\text{O}_2}^{-x}$

with $x$ the Brouwer exponent (`TRANSF_EXPONENT`). The expression is linear in the
three conductivities, so it is solved by non-negative least squares, which is what
keeps all three from going negative.

The ionic transference number then follows at every $p_{\text{O}_2}$:
$t_{\text{ion}} = \sigma_{\text{ion}}/\sigma_{\text{tot}}$ and
$t_{\text{elect}} = 1 - t_{\text{ion}}$. The local slope of the Brouwer curve is
$x\,(t_{p\text{-type}} - t_{n\text{-type}})$, so a plateau reads as purely ionic, a
slope of $+x$ as purely p-type and a slope of $-x$ as purely n-type. Derivation:
`docs/MATHEMATICS.md` section 5.

The table is exported and one figure is drawn for **every** peak.


In [ ]:
# Transference numbers: table for all peaks, figure for BROUWER_PEAK_ID
from pipeline.plots import (fit_transference, plot_brouwer_transference,
                            plot_transference_arrhenius)

if "df_all_peaks" not in globals():
    print("[SKIP] transference: run Step 1 and Step 2 first (df_all_peaks not in memory)")
elif not (PO2_PROBE and "pO2_mean" in df_all_peaks.columns
          and df_all_peaks["pO2_mean"].notna().any()):
    print("[SKIP] transference: no usable pO2 data (probe off, or none recorded)")
else:
    brouwer_dir = RESULTS_BASE / "pO2"
    _t_tables = []
    for _pid in sorted(df_all_peaks["peak_id"].unique()):
        _df_t = fit_transference(df_all_peaks, peak_id=int(_pid),
                                 exponent=TRANSF_EXPONENT, temps=BROUWER_TEMPS)
        if not _df_t.empty:
            _t_tables.append(_df_t)
    if not _t_tables:
        print("[SKIP] transference: not enough pO2 points per temperature")
    else:
        df_transference = pd.concat(_t_tables, ignore_index=True)
        _xlsx = brouwer_dir / "stage4_transference.xlsx"
        try:
            with pd.ExcelWriter(_xlsx, engine="openpyxl") as _w:
                df_transference.to_excel(_w, sheet_name="Transference", index=False)
            print(f"Transference table saved: {_xlsx.relative_to(NOTEBOOK_DIR)}")
        except Exception as _exc:
            print(f"[WARN] could not write {_xlsx.name}: {type(_exc).__name__}: {_exc}")

        # one figure per process; figures restricted to TRANSF_PEAK_IDS
        _fig_pids = (sorted(df_transference["peak_id"].unique())
                     if TRANSF_PEAK_IDS is None else list(TRANSF_PEAK_IDS))
        for _pid in _fig_pids:
            print(f"Peak {_pid}:")
            try:
                fig_tr = plot_brouwer_transference(
                    df_all_peaks, save_dir=brouwer_dir, sample_name=sample_id,
                    peak_id=int(_pid), exponent=TRANSF_EXPONENT,
                    temps_to_plot=BROUWER_TEMPS)
                plt.show()
                plt.close(fig_tr)
            except ValueError as _exc:
                print(f"  [SKIP] {_exc}")
            # Arrhenius of the partial conductivities: the rigorous check
            # that sigma_ion and sigma_p are two distinct activated channels
            fig_ta = plot_transference_arrhenius(
                df_transference, save_dir=brouwer_dir,
                sample_name=sample_id, peak_id=int(_pid))
            if fig_ta is not None:
                plt.show()
                plt.close(fig_ta)

        # one row per (peak, T): fitted components and the t_ion range over pO2
        _sum = (df_transference
                .groupby(["peak_id", "T_nominal"])
                .agg(sigma_ion=("sigma_ion", "first"), sigma_p=("sigma_p", "first"),
                     sigma_n=("sigma_n", "first"), R2=("R2", "first"),
                     t_ion_min=("t_ion", "min"), t_ion_max=("t_ion", "max"))
                .reset_index())
        display(_sum.round(4))


## Output summary

Figures land in the sub-folders of `Results/{condition}/`:

- `DRT/`: stacked $\gamma(\log\tau)$, one curve per temperature
- `Nyquist-Bode/`: Nyquist overlay and Bode
- `Arrhenius/`: the 2x2 Arrhenius panel
- `pO2/`: the Brouwer $p_{\text{O}_2}$ diagram, all conditions together

Every figure is written twice: PNG for a quick look, PDF for publication.

**Next step:** run [stage5_model.ipynb](stage5_model.ipynb)
